In [27]:
### Parcsing Aseg.stats file and saving to CSV

import pandas as pd
from pathlib import Path

# Pick one aseg.stats file from the new FreeSurfer output
base_dir = Path(r"C:\Users\okkam\Desktop\free_surfer_output")
stats_files = sorted((base_dir / "aseg").rglob("*_aseg.stats"))
if not stats_files:
    raise FileNotFoundError(f"No aseg.stats files found under {base_dir / 'aseg'}")

stats_file = stats_files[0]
csv_output = stats_file.with_name(f"{stats_file.stem}_parsed.csv")

# Read the file and extract the data table
with open(stats_file, 'r') as f:
    lines = f.readlines()

# Extract ICV (eTIV) from header
icv = None
for line in lines:
    if 'EstimatedTotalIntraCranialVol' in line and line.startswith('# Measure'):
        parts = line.split(',')
        if len(parts) >= 4:
            icv = float(parts[3].strip())
            break

# Find the line with column headers
header_line_idx = None
for i, line in enumerate(lines):
    if line.startswith('# ColHeaders'):
        header_line_idx = i
        break

# Extract column names from the header line
header_line = lines[header_line_idx].strip()
columns = header_line.replace('# ColHeaders ', '').split()

# Find where data starts (after the header line)
data_start_idx = header_line_idx + 1

# Parse the data rows
data_rows = []
for line in lines[data_start_idx:]:
    if line.strip() and not line.startswith('#'):
        values = line.strip().split()
        if len(values) == len(columns):
            data_rows.append(values)

# Create DataFrame
df = pd.DataFrame(data_rows, columns=columns)

# Add ICV column
if icv is not None:
    df['ICV'] = icv

# Convert numeric columns to appropriate types
numeric_cols = ['Index', 'SegId', 'NVoxels', 'Volume_mm3', 'normMean', 'normStdDev', 'normMin', 'normMax', 'normRange', 'ICV']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Save to CSV
df.to_csv(csv_output, index=False)

print(f"✓ Parsed aseg.stats file")
print(f"  Rows: {len(df)}")
print(f"  Columns: {', '.join(df.columns)}")
print(f"  ICV (eTIV): {icv:.1f} mm³")
print(f"\n✓ Saved to: {csv_output}")
print(f"\nFirst few rows:")
print(df.head())

✓ Parsed aseg.stats file
  Rows: 45
  Columns: Index, SegId, NVoxels, Volume_mm3, StructName, normMean, normStdDev, normMin, normMax, normRange, ICV
  ICV (eTIV): 1256608.4 mm³

✓ Saved to: C:\Users\okkam\Desktop\free_surfer_output\aseg\sub-3002498_ses-t02_aseg_parsed.csv

First few rows:
   Index  SegId  NVoxels  Volume_mm3                    StructName  normMean  \
0      1      4    10051     10191.8        Left-Lateral-Ventricle   21.2026   
1      2      5      305       335.4             Left-Inf-Lat-Vent   36.8557   
2      3      7    12007     12859.4  Left-Cerebellum-White-Matter   84.5708   
3      4      8    38338     38234.8        Left-Cerebellum-Cortex   56.0661   
4      5     10     6476      6312.9                 Left-Thalamus   85.3816   

   normStdDev  normMin  normMax  normRange           ICV  
0     11.8863      0.0     72.0       72.0  1.256608e+06  
1     14.4377      0.0     74.0       74.0  1.256608e+06  
2      7.3507     45.0    115.0       70.0  1.256608

In [ ]:
### Batch parse all FreeSurfer stats files and save CSVs

import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Root folders that contain the aseg and aparc outputs
base_dir = Path(r"C:\Users\okkam\Desktop\free_surfer_output")
aseg_dir = base_dir / "aseg"
aparc_dir = base_dir / "aparc"


def parse_aseg_stats(stats_file):
    """Parse an aseg.stats file and return a DataFrame."""
    with open(stats_file, "r") as f:
        lines = f.readlines()

    icv = None
    for line in lines:
        if "EstimatedTotalIntraCranialVol" in line and line.startswith("# Measure"):
            parts = line.split(",")
            if len(parts) >= 4:
                icv = float(parts[3].strip())
                break

    header_line_idx = None
    for i, line in enumerate(lines):
        if line.startswith("# ColHeaders"):
            header_line_idx = i
            break

    if header_line_idx is None:
        return None

    columns = lines[header_line_idx].strip().replace("# ColHeaders ", "").split()
    data_rows = []

    for line in lines[header_line_idx + 1 :]:
        if line.strip() and not line.startswith("#"):
            values = line.strip().split()
            if len(values) == len(columns):
                data_rows.append(values)

    if not data_rows:
        return None

    df = pd.DataFrame(data_rows, columns=columns)

    if icv is not None:
        df["ICV"] = icv

    numeric_cols = ["Index", "SegId", "NVoxels", "Volume_mm3", "normMean", "normStdDev", "normMin", "normMax", "normRange", "ICV"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def parse_aparc_stats(stats_file):
    """Parse an aparc.stats file and return a DataFrame."""
    with open(stats_file, "r") as f:
        lines = f.readlines()

    measures = {}
    for line in lines:
        if line.startswith("# Measure"):
            parts = line.split(",")
            if len(parts) >= 4:
                measure_name = parts[0].replace("# Measure ", "").strip()
                measure_value = parts[3].strip()
                try:
                    measures[measure_name] = float(measure_value)
                except ValueError:
                    measures[measure_name] = measure_value

    header_line_idx = None
    for i, line in enumerate(lines):
        if line.startswith("# ColHeaders"):
            header_line_idx = i
            break

    if header_line_idx is None:
        return None

    columns = lines[header_line_idx].strip().replace("# ColHeaders ", "").split()
    data_rows = []

    for line in lines[header_line_idx + 1 :]:
        if line.strip() and not line.startswith("#"):
            values = line.strip().split()
            if len(values) >= len(columns):
                data_rows.append([values[0]] + values[1 : len(columns)])

    if not data_rows:
        return None

    df = pd.DataFrame(data_rows, columns=columns)

    numeric_cols = ["NumVert", "SurfArea", "GrayVol", "ThickAvg", "ThickStd", "MeanCurv", "GausCurv", "FoldInd", "CurvInd"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    for measure_name, measure_value in measures.items():
        if isinstance(measure_value, (int, float)):
            df[measure_name] = measure_value

    return df


# Collect all stats files from both folders
stats_files = []
stats_files.extend(sorted(aseg_dir.rglob("*_aseg.stats")))
stats_files.extend(sorted(aparc_dir.rglob("*_lh.aparc.stats")))
stats_files.extend(sorted(aparc_dir.rglob("*_rh.aparc.stats")))
stats_files = [stats_file for stats_file in stats_files if stats_file.exists()]

print(f"Found {len(stats_files)} stats files to parse\n")

processed = 0
errors = 0

for stats_file in tqdm(stats_files, desc="Parsing stats files"):
    try:
        if stats_file.name.endswith("_aseg.stats"):
            df = parse_aseg_stats(stats_file)
        else:
            df = parse_aparc_stats(stats_file)

        if df is None:
            errors += 1
            print(f"✗ Failed to parse: {stats_file}")
            continue

        csv_output = stats_file.with_name(f"{stats_file.stem}_parsed.csv")
            df.to_csv(csv_output, index=False)
            processed += 1
    except Exception as e:
        errors += 1
        print(f"✗ Error processing {stats_file}: {e}")

print(f"\n✓ Processing complete!")
print(f"  Successfully processed: {processed}")
print(f"  Errors: {errors}")

Found 65 aseg.stats files



Processing aseg.stats files: 100%|██████████| 65/65 [00:04<00:00, 13.75it/s]


✓ Processing complete!
  Successfully processed: 65
  Errors: 0


In [ ]:
##### aparc processing

import pandas as pd
from pathlib import Path

# Files to parse
lh_file = Path(r"C:\Users\okkam\Desktop\lh.aparc.stats")
rh_file = Path(r"C:\Users\okkam\Desktop\rh.aparc.stats")

def parse_aparc_stats(stats_file):
    """Parse aparc.stats file and extract header measures and region data"""
    with open(stats_file, 'r') as f:
        lines = f.readlines()
    
    # Extract header measures
    measures = {}
    for line in lines:
        if line.startswith('# Measure'):
            parts = line.split(',')
            if len(parts) >= 4:
                measure_name = parts[0].replace('# Measure ', '').strip()
                measure_value = parts[3].strip()
                try:
                    measures[measure_name] = float(measure_value)
                except:
                    measures[measure_name] = measure_value
    
    # Find the line with column headers
    col_header_line_idx = None
    for i, line in enumerate(lines):
        if line.startswith('# ColHeaders'):
            col_header_line_idx = i
            break
    
    if col_header_line_idx is None:
        return None, None
    
    # Extract column names
    header_line = lines[col_header_line_idx].strip()
    columns = header_line.replace('# ColHeaders ', '').split()
    
    # Find where data starts
    data_start_idx = col_header_line_idx + 1
    
    # Parse the data rows
    data_rows = []
    for line in lines[data_start_idx:]:
        if line.strip() and not line.startswith('#'):
            values = line.strip().split()
            if len(values) >= len(columns):
                # Handle structure name which might have spaces (take first column as name)
                struct_name = values[0]
                data_values = [struct_name] + values[1:len(columns)]
                data_rows.append(data_values)
    
    # Create DataFrame
    df = pd.DataFrame(data_rows, columns=columns)
    
    # Convert numeric columns
    numeric_cols = ['NumVert', 'SurfArea', 'GrayVol', 'ThickAvg', 'ThickStd', 'MeanCurv', 'GausCurv', 'FoldInd', 'CurvInd']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Add key measures as columns
    for measure_name, measure_value in measures.items():
        if isinstance(measure_value, (int, float)):
            df[measure_name] = measure_value
    
    return df, measures

# Parse both files
print("Parsing lh.aparc.stats...")
lh_df, lh_measures = parse_aparc_stats(lh_file)

print("Parsing rh.aparc.stats...")
rh_df, rh_measures = parse_aparc_stats(rh_file)

if lh_df is not None:
    lh_csv = lh_file.parent / "lh.aparc_parsed.csv"
    lh_df.to_csv(lh_csv, index=False)
    print(f"✓ Saved lh: {lh_csv}")
    print(f"  Regions: {len(lh_df)}")
    print(f"  Columns: {', '.join(lh_df.columns.tolist()[:10])}...")

if rh_df is not None:
    rh_csv = rh_file.parent / "rh.aparc_parsed.csv"
    rh_df.to_csv(rh_csv, index=False)
    print(f"✓ Saved rh: {rh_csv}")
    print(f"  Regions: {len(rh_df)}")
    print(f"  Columns: {', '.join(rh_df.columns.tolist()[:10])}...")

print("\n✓ Done! Both aparc files parsed and saved.")

Parsing lh.aparc.stats...
Parsing rh.aparc.stats...
✓ Saved lh: C:\Users\okkam\Desktop\lh.aparc_parsed.csv
  Regions: 34
  Columns: StructName, NumVert, SurfArea, GrayVol, ThickAvg, ThickStd, MeanCurv, GausCurv, FoldInd, CurvInd...
✓ Saved rh: C:\Users\okkam\Desktop\rh.aparc_parsed.csv
  Regions: 34
  Columns: StructName, NumVert, SurfArea, GrayVol, ThickAvg, ThickStd, MeanCurv, GausCurv, FoldInd, CurvInd...

✓ Done! Both aparc files parsed and saved.


In [5]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Path to T1 folder with all participants
t1_folder = Path(r"D:\02-Raw_data-anat\longitudinal_freesurfer_149\longi_output\T2")

# Find all lh and rh aparc.stats files
lh_aparc_files = list(t1_folder.rglob("lh.aparc.stats"))
rh_aparc_files = list(t1_folder.rglob("rh.aparc.stats"))

print(f"Found {len(lh_aparc_files)} lh.aparc.stats files")
print(f"Found {len(rh_aparc_files)} rh.aparc.stats files\n")

def parse_aparc_stats(stats_file):
    """Parse aparc.stats file and extract header measures and region data"""
    with open(stats_file, 'r') as f:
        lines = f.readlines()
    
    # Extract header measures
    measures = {}
    for line in lines:
        if line.startswith('# Measure'):
            parts = line.split(',')
            if len(parts) >= 4:
                measure_name = parts[0].replace('# Measure ', '').strip()
                measure_value = parts[3].strip()
                try:
                    measures[measure_name] = float(measure_value)
                except:
                    measures[measure_name] = measure_value
    
    # Find the line with column headers
    col_header_line_idx = None
    for i, line in enumerate(lines):
        if line.startswith('# ColHeaders'):
            col_header_line_idx = i
            break
    
    if col_header_line_idx is None:
        return None, None
    
    # Extract column names
    header_line = lines[col_header_line_idx].strip()
    columns = header_line.replace('# ColHeaders ', '').split()
    
    # Find where data starts
    data_start_idx = col_header_line_idx + 1
    
    # Parse the data rows
    data_rows = []
    for line in lines[data_start_idx:]:
        if line.strip() and not line.startswith('#'):
            values = line.strip().split()
            if len(values) >= len(columns):
                struct_name = values[0]
                data_values = [struct_name] + values[1:len(columns)]
                data_rows.append(data_values)
    
    # Create DataFrame
    df = pd.DataFrame(data_rows, columns=columns)
    
    # Convert numeric columns
    numeric_cols = ['NumVert', 'SurfArea', 'GrayVol', 'ThickAvg', 'ThickStd', 'MeanCurv', 'GausCurv', 'FoldInd', 'CurvInd']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Add key measures as columns
    for measure_name, measure_value in measures.items():
        if isinstance(measure_value, (int, float)):
            df[measure_name] = measure_value
    
    return df, measures

# Process all lh and rh aparc files
processed = 0
errors = 0
all_aparc_files = lh_aparc_files + rh_aparc_files

for aparc_file in tqdm(all_aparc_files, desc="Processing aparc.stats files"):
    try:
        df, measures = parse_aparc_stats(aparc_file)
        
        if df is not None:
            # Determine output filename based on lh or rh
            if 'lh.aparc' in aparc_file.name:
                csv_output = aparc_file.parent / "lh.aparc_parsed.csv"
            else:
                csv_output = aparc_file.parent / "rh.aparc_parsed.csv"
            
            df.to_csv(csv_output, index=False)
            processed += 1
        else:
            errors += 1
            print(f"✗ Failed to parse: {aparc_file}")
    except Exception as e:
        errors += 1
        print(f"✗ Error processing {aparc_file}: {e}")

print(f"\n✓ Processing complete!")
print(f"  Successfully processed: {processed}")
print(f"  Errors: {errors}")

Found 65 lh.aparc.stats files
Found 65 rh.aparc.stats files



Processing aparc.stats files: 100%|██████████| 130/130 [00:08<00:00, 14.74it/s]


✓ Processing complete!
  Successfully processed: 130
  Errors: 0


In [31]:
### Combine all parsed FreeSurfer outputs into one CSV

import pandas as pd
from pathlib import Path
from tqdm import tqdm
import re

# New FreeSurfer output folders
base_dir = Path(r"C:\Users\okkam\Desktop\free_surfer_output")
aseg_dir = base_dir / "aseg"
aparc_dir = base_dir / "aparc"

# Find all parsed CSVs created by the parsing cells
aseg_csvs = sorted(aseg_dir.rglob("*_aseg_parsed.csv"))
lh_csvs = sorted(aparc_dir.rglob("*_lh.aparc_parsed.csv"))
rh_csvs = sorted(aparc_dir.rglob("*_rh.aparc_parsed.csv"))

print(f"Found {len(aseg_csvs)} aseg parsed files")
print(f"Found {len(lh_csvs)} lh aparc parsed files")
print(f"Found {len(rh_csvs)} rh aparc parsed files\n")

# Function to extract participant ID and visit from file name
def extract_participant_info(file_name):
    match_id = re.search(r'sub-(\d+)', file_name)
    if not match_id:
        match_id = re.search(r'^(\d+)_', file_name)
    participant_id = match_id.group(1) if match_id else "unknown"

    match_visit = re.search(r'_ses-(t\d+)', file_name)
    visit = match_visit.group(1) if match_visit else "unknown"

    return participant_id, visit

# Function to flatten aseg data - keep only Volume_mm3 and ICV
def flatten_aseg(aseg_df):
    flattened = {}
    for idx, row in aseg_df.iterrows():
        struct_name = str(row['StructName']).replace('-', '_').replace(' ', '_')
        if 'Volume_mm3' in aseg_df.columns:
            flattened[f"aseg_{struct_name}_Volume_mm3"] = row['Volume_mm3']
        if idx == 0 and 'ICV' in aseg_df.columns:
            icv_values = pd.to_numeric(aseg_df['ICV'], errors='coerce').dropna()
            if not icv_values.empty:
                flattened['aseg_ICV'] = icv_values.iloc[0]
    return flattened

def extract_aseg_icv_from_stats(stats_path):
    if not stats_path.exists():
        return None

    with open(stats_path, 'r') as f:
        for line in f:
            if line.startswith('# Measure') and 'EstimatedTotalIntraCranialVol' in line:
                parts = [part.strip() for part in line.split(',')]
                for part in reversed(parts):
                    try:
                        return float(part)
                    except ValueError:
                        continue
    return None

# Function to flatten aparc data - keep only Thickness and SurfArea
def flatten_aparc(aparc_df, hemisphere):
    flattened = {}
    for _, row in aparc_df.iterrows():
        region_name = str(row['StructName']).replace('-', '_').replace(' ', '_')
        if 'ThickAvg' in aparc_df.columns:
            flattened[f"{hemisphere}_{region_name}_thickness"] = row['ThickAvg']
        if 'GrayVol' in aparc_df.columns:
            flattened[f"{hemisphere}_{region_name}_GrayVol"] = row['GrayVol']
        if 'SurfArea' in aparc_df.columns:
            flattened[f"{hemisphere}_{region_name}_SurfArea"] = row['SurfArea']
    return flattened

# Load and combine parsed files by subject/session
all_data = []
errors = 0

def csv_key(path_obj):
    return path_obj.name.replace('_aseg_parsed.csv', '').replace('_lh.aparc_parsed.csv', '').replace('_rh.aparc_parsed.csv', '')

aseg_map = {csv_key(path): path for path in aseg_csvs}
lh_map = {csv_key(path): path for path in lh_csvs}
rh_map = {csv_key(path): path for path in rh_csvs}
common_keys = sorted(set(aseg_map) & set(lh_map) & set(rh_map))

print(f"Found {len(common_keys)} complete participants with all 3 parsed files\n")

for key in tqdm(common_keys, desc="Combining participants"):
    try:
        participant_id, visit = extract_participant_info(key)

        aseg_df = pd.read_csv(aseg_map[key])
        lh_aparc_df = pd.read_csv(lh_map[key])
        rh_aparc_df = pd.read_csv(rh_map[key])
        raw_aseg_path = aseg_map[key].with_name(aseg_map[key].name.replace('_aseg_parsed.csv', '_aseg.stats'))

        row_data = {
            'participant_id': participant_id,
            'visit': visit
        }

        row_data.update(flatten_aseg(aseg_df))
        if 'aseg_ICV' not in row_data or pd.isna(row_data.get('aseg_ICV')):
            raw_icv = extract_aseg_icv_from_stats(raw_aseg_path)
            if raw_icv is not None:
                row_data['aseg_ICV'] = raw_icv
        row_data.update(flatten_aparc(lh_aparc_df, 'lh'))
        row_data.update(flatten_aparc(rh_aparc_df, 'rh'))
        all_data.append(row_data)

    except Exception as e:
        errors += 1
        print(f"✗ Error processing {key}: {e}")

# Create combined DataFrame
if all_data:
    combined_df = pd.DataFrame(all_data)
    output_path = base_dir / "all_participants_freesurfer_data.csv"
    combined_df.to_csv(output_path, index=False)

    print(f"\n✓ Combined data saved!")
    print(f"  Participants: {len(combined_df)}")
    print(f"  Total Variables: {len(combined_df.columns)}")
    print(f"  Output: {output_path}")
    print(f"\nVariable breakdown:")
    print(f"  - Base columns (participant_id, visit): 2")
    print(f"  - ASEG volumes + ICV")
    print(f"  - LH APARC Thickness + GrayVol + SurfArea")
    print(f"  - RH APARC Thickness + GrayVol + SurfArea")
    print(f"\nSample columns:")
    print(combined_df.columns.tolist()[:20])
    print(f"\nFirst few rows:")
    print(combined_df.iloc[:3, :6])
    print(f"\nErrors: {errors}")
else:
    print("✗ No complete parsed data to combine")

Found 267 aseg parsed files
Found 267 lh aparc parsed files
Found 267 rh aparc parsed files

Found 267 complete participants with all 3 parsed files



Combining participants: 100%|██████████| 267/267 [00:10<00:00, 25.63it/s]



✓ Combined data saved!
  Participants: 267
  Total Variables: 252
  Output: C:\Users\okkam\Desktop\free_surfer_output\all_participants_freesurfer_data.csv

Variable breakdown:
  - Base columns (participant_id, visit): 2
  - ASEG volumes + ICV
  - LH APARC Thickness + GrayVol + SurfArea
  - RH APARC Thickness + GrayVol + SurfArea

Sample columns:
['participant_id', 'visit', 'aseg_Left_Lateral_Ventricle_Volume_mm3', 'aseg_ICV', 'aseg_Left_Inf_Lat_Vent_Volume_mm3', 'aseg_Left_Cerebellum_White_Matter_Volume_mm3', 'aseg_Left_Cerebellum_Cortex_Volume_mm3', 'aseg_Left_Thalamus_Volume_mm3', 'aseg_Left_Caudate_Volume_mm3', 'aseg_Left_Putamen_Volume_mm3', 'aseg_Left_Pallidum_Volume_mm3', 'aseg_3rd_Ventricle_Volume_mm3', 'aseg_4th_Ventricle_Volume_mm3', 'aseg_Brain_Stem_Volume_mm3', 'aseg_Left_Hippocampus_Volume_mm3', 'aseg_Left_Amygdala_Volume_mm3', 'aseg_CSF_Volume_mm3', 'aseg_Left_Accumbens_area_Volume_mm3', 'aseg_Left_VentralDC_Volume_mm3', 'aseg_Left_vessel_Volume_mm3']

First few rows:
  p

In [33]:
from pathlib import Path
import pandas as pd

input_csv = Path(r"C:\Users\okkam\Desktop\free_surfer_output\all_participants_freesurfer_data.csv")
output_csv = input_csv.with_name(f"{input_csv.stem}_selected_variables.csv")

selected_columns = [
    "participant_id",
    "visit",
    "aseg_ICV",
    "aseg_Left_Hippocampus_Volume_mm3",
    "aseg_Right_Hippocampus_Volume_mm3",
    "lh_entorhinal_thickness",
    "lh_entorhinal_SurfArea",
    "lh_parahippocampal_thickness",
    "lh_parahippocampal_SurfArea",
    "rh_entorhinal_thickness",
    "rh_entorhinal_SurfArea",
    "rh_parahippocampal_thickness",
    "rh_parahippocampal_SurfArea",
    "lh_inferiortemporal_thickness",
    "lh_inferiortemporal_SurfArea",
    "rh_inferiortemporal_thickness",
    "rh_inferiortemporal_SurfArea",
    "lh_temporalpole_thickness",
    "lh_temporalpole_SurfArea",
    "rh_temporalpole_thickness",
    "rh_temporalpole_SurfArea",
    "lh_inferiorparietal_thickness",
    "lh_inferiorparietal_SurfArea",
    "rh_inferiorparietal_thickness",
    "rh_inferiorparietal_SurfArea",
    "lh_superiorfrontal_thickness",
    "lh_superiorfrontal_SurfArea",
    "rh_superiorfrontal_thickness",
    "rh_superiorfrontal_SurfArea",
    "lh_superiorparietal_thickness",
    "lh_superiorparietal_SurfArea",
    "rh_superiorparietal_thickness",
    "rh_superiorparietal_SurfArea",
    "lh_supramarginal_thickness",
    "lh_supramarginal_SurfArea",
    "rh_supramarginal_thickness",
    "rh_supramarginal_SurfArea",
    "lh_precuneus_thickness",
    "lh_precuneus_SurfArea",
    "rh_precuneus_thickness",
    "rh_precuneus_SurfArea",
    "lh_pericalcarine_thickness",
    "lh_pericalcarine_SurfArea",
    "rh_pericalcarine_thickness",
    "rh_pericalcarine_SurfArea",
    "lh_parsopercularis_thickness",
    "lh_parsorbitalis_thickness",
    "lh_parstriangularis_thickness",
    "lh_parsopercularis_SurfArea",
    "lh_parsorbitalis_SurfArea",
    "lh_parstriangularis_SurfArea",
    "rh_parsopercularis_thickness",
    "rh_parsorbitalis_thickness",
    "rh_parstriangularis_thickness",
    "rh_parsopercularis_SurfArea",
    "rh_parsorbitalis_SurfArea",
    "rh_parstriangularis_SurfArea",
]

df = pd.read_csv(input_csv)
missing_columns = [column for column in selected_columns if column not in df.columns]
available_columns = [column for column in selected_columns if column in df.columns]

if missing_columns:
    print(f"Missing columns in input file: {missing_columns}")
    print("Saving the columns that are available in the source file.")

filtered_df = df[available_columns].copy()
filtered_df.to_csv(output_csv, index=False)

print(f"Saved filtered file to: {output_csv}")
print(f"Kept {len(available_columns)} columns and {len(filtered_df)} rows")

Saved filtered file to: C:\Users\okkam\Desktop\free_surfer_output\all_participants_freesurfer_data_selected_variables.csv
Kept 57 columns and 267 rows
